In [ ]:
# ============================================================
# PARAMETERS (papermill injects these)
# ============================================================

config_path = None
run_dir = None

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import json
import yaml
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

from pathlib import Path

from collections import Counter

from scripts.set_seed import set_seed

from src.model_factory import build_model

from src.embedding_registry import get_embedding_dir

from src.embeddings import load_single_embeddings_from_manifest

from src.training import (
    train_classifier,
    make_weighted_ce,
    print_final_training_summary,
    summarize_final_in_sample_metrics,
    final_in_sample_classification_table,
    plot_final_training_history,
    plot_final_macro_metrics,
    plot_final_confusion_matrix,
    plot_final_roc_curves,
    evaluate_split
)

In [ ]:
# ============================================================
# LOAD CONFIG, SEED, AND MODEL
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

RUN_DIR = Path(run_dir)
RUN_DIR.mkdir(parents=True, exist_ok=True)

set_seed(cfg["experiment"]["seed"])

model = build_model(cfg)

print(json.dumps(cfg, indent=2))

In [ ]:
# ============================================================
# LOAD EMBEDDINGS
# ============================================================

evaluation_mode = cfg["evaluation"]["mode"]

print(f"Evaluation mode: {evaluation_mode}")

embedding_dir = get_embedding_dir(cfg)

embeddings_dict = load_single_embeddings_from_manifest(manifest_candidates=[embedding_dir])

In [ ]:
# ============================================================
# FINAL MODEL TRAINING
# ============================================================

# Prepare all-data loaders 
train_ds = TensorDataset(embeddings_dict['train_embeddings'], embeddings_dict['train_labels'])
val_ds = TensorDataset(embeddings_dict['val_embeddings'], embeddings_dict['val_labels'])

if cfg["hyperparameters"]["use_custom_hyperparameters"]:
    best_hp = cfg["hyperparameters"]
    class_counts_all = Counter(embeddings_dict['train_labels'].numpy())
    counts_list_all  = [class_counts_all.get(i, 1) for i in range(3)]

    if best_hp["useWeightedSampler"]:
        sample_weights_all = torch.tensor(
            [1.0 / class_counts_all.get(int(l), 1) for l in embeddings_dict['train_labels'].numpy()],
            dtype=torch.float,
        )
        sampler_all = WeightedRandomSampler(
            sample_weights_all, len(sample_weights_all), replacement=True,
        )
        final_train_loader = DataLoader(
            train_ds,
            batch_size=best_hp["batch_size"],
            sampler=sampler_all,
            drop_last=True,
        )
    else:
        final_train_loader = DataLoader(
            train_ds,
            batch_size=best_hp["batch_size"],
            shuffle=True,
            drop_last=True,
        )
    # Val loader = train loader (in-sample monitoring only, no early stopping)
    final_val_loader = DataLoader(
        val_ds,
        batch_size=best_hp["batch_size"],
        shuffle=False,
        drop_last=False,
        )

    criterion = (
        make_weighted_ce(counts_list_all, device, power=best_hp.get("weighted_ce_power", 1.0))
        if best_hp.get("weighted_ce_power", -1.0) >= 0.0
        else None
    )

else: 
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
    criterion = None
    best_hp = {}

# Train model
final_model = model().to(device)

history = train_classifier(
    final_model,
    train_loader,
    val_loader,
    num_epochs=50,
    device=device,
    patience=50,
    criterion=criterion,
    optimizer_lr=best_hp.get("optimizer_lr", 1e-3),
    optimizer_weight_decay=best_hp.get("optimizer_weight_decay", 1e-2),
    schedular_patience=best_hp.get("schedular_patience", 10),
    warmup_epochs=best_hp.get("warmup_epochs", 5),
    checkpoint_path=str(RUN_DIR / "model_best_epoch.pt")
)

In [ ]:
# Display tables
print_final_training_summary(history_final, save_path=RUN_DIR / "final_training", insample=False)
summarize_final_in_sample_metrics(history_final, class_names=['barrier', 'cation', 'anion'], save_path=RUN_DIR / "final_training", insample=False)
final_in_sample_classification_table(history_final, class_names=['barrier', 'cation', 'anion'], save_path=RUN_DIR / "final_training", insample=False)

# Display plots
plot_final_training_history(history_final, save_path=RUN_DIR / "final_training", insample=False)
plot_final_macro_metrics(history_final, save_path=RUN_DIR / "final_training", insample=False)
plot_final_confusion_matrix(history_final, class_names=['barrier', 'cation', 'anion'], save_path=RUN_DIR / "final_training", insample=False)
plot_final_roc_curves(history_final, class_names=['barrier', 'cation', 'anion'], save_path=RUN_DIR / "final_training", insample=False)

In [ ]:
# ============================================================
# SAVE FINAL MODEL
# ============================================================

torch.save(
    final_model.state_dict(),
    RUN_DIR / "final_model.pt",
)

print("Final model saved.")

In [ ]:
# ============================================================
# SAVE FINAL METADATA
# ============================================================

final_metadata = {
    "experiment_name": cfg["experiment"]["name"],
    "evaluation_mode": evaluation_mode,
}

with open(
    RUN_DIR / "training_metadata.json",
    "w",
) as f:
    json.dump(final_metadata, f, indent=2)

print("Training complete.")